# **Portfolio:** Mario Casanova — Data Science & Analytics Portfolio
## **Case Study:** Thermodynamics & In Silico Biology (Ramachandran Plot)

---
*Nota Institucional: Este notebook representa la deconstrucción matemática de la biología estructural. En lugar de depender de paquetes químicos pre-compilados, traduce las restricciones físicas de la materia en un mapeo de densidad bidimensional calculable directamente mediante álgebra lineal y cálculo vectorial (productos cruzados, normatización, y arcos tangentes bidimensionales). Transformar la biología en ciencia de datos geométrica pura.*

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math

plt.style.use('dark_background')
sns.set_palette("magma")
np.seterr(all='ignore') # Ignorar warnings vectoriales de divisiones nulas

### 1. Ingestión de Coordenadas Atómicas Brutas
De acuerdo al postulado de Ramachandran, la conformación topológica de una proteína está dictada EXCLUSIVAMENTE por dos de los tres enlaces repetitivos en el esqueleto celular: el ángulo $\phi$ *(Phi)* y el ángulo $\psi$ *(Psi)*. El enlace peptídico $\omega$ *(Omega)* es esencialmente plano por su naturaleza de doble-enlace parcial.

Extraemos las secuencias fijas de los tres átomos vertebrales estructurales: Nitrógeno (N), Carbono Alfa (CA) y Carbono Carboxilo (C).

In [ ]:
def extract_backbone(filepath):
    """
    Parseamos secuencialmente los átomos requeridos para 
    componer los planos de torsión.
    """
    atoms = []
    with open(filepath, 'r') as f:
        for line in f:
            if line.startswith('ATOM'):
                atom_name = line[12:16].strip()
                if atom_name in ['N', 'CA', 'C']:
                    res_num = int(line[22:26].strip())
                    res_name = line[17:20].strip()
                    x, y, z = float(line[30:38]), float(line[38:46]), float(line[46:54])
                    
                    atoms.append({
                        'Residue': res_num,
                        'Name': res_name,
                        'Atom': atom_name,
                        'Coors': np.array([x, y, z])
                    })
    return pd.DataFrame(atoms)

df_backbone = extract_backbone('../data/1aho.pdb')
print(f"[*] Esqueleto extraído: {len(df_backbone)} vectores atómicos.")
df_backbone.head()

### 2. Motor Algorítmico Diédrico (Cálculo Vectorial)
El cálculo del ángulo diédrico entre cuatro átomos $(p_0, p_1, p_2, p_3)$ formados por tres vectores de enlace $(b_1, b_2, b_3)$ se formaliza como el ángulo entre el plano inclinado por $b_1$ y $b_2$, vs el plano por $b_2$ y $b_3$.

$\text{Dih}(p_1, p_2, p_3, p_4) = \text{atan2}(|b_2| b_1 \cdot (b_2 \times b_3), (b_1 \times b_2) \cdot (b_2 \times b_3))$

In [ ]:
def compute_dihedral(p0, p1, p2, p3):
    """
    Cálculo en determinismo geométrico puro del ángulo de torsión
    entre 4 coordenadas tridimensionales en el espacio Euclideo.
    """
    b1 = p1 - p0
    b2 = p2 - p1
    b3 = p3 - p2
    
    n1 = np.cross(b1, b2) # Vector normal del primer plano
    n2 = np.cross(b2, b3) # Vector normal del segundo plano
    
    # Proyecciones ortogonales para la arcotangente direccional (atan2)
    m1 = np.cross(n1, b2 / np.linalg.norm(b2))
    
    x = np.dot(n1, n2)
    y = np.dot(m1, n2)
    
    return np.degrees(np.arctan2(y, x))

def build_ramachandran(df):
    """
    Orquestador vectorial. Itera la secuencia construyendo los
    cuartetos atómicos requeridos para medir Phi y Psi. 
    """
    phi_angles = []
    psi_angles = []
    
    # Los residuos cambian secuencialmente.
    residues = df['Residue'].unique()
    
    for i in range(1, len(residues) - 1): # Excluimos primero y último para tener enlaces completos
        r_prev = residues[i-1]
        r_curr = residues[i]
        r_next = residues[i+1]
        
        try:
            # Coordenadas Phi (C_prev, N, CA, C)
            C_prev = df[(df.Residue == r_prev) & (df.Atom == 'C')]['Coors'].values[0]
            N_curr = df[(df.Residue == r_curr) & (df.Atom == 'N')]['Coors'].values[0]
            CA_curr = df[(df.Residue == r_curr) & (df.Atom == 'CA')]['Coors'].values[0]
            C_curr = df[(df.Residue == r_curr) & (df.Atom == 'C')]['Coors'].values[0]
            
            # Coordenadas Psi (N, CA, C, N_next)
            N_next = df[(df.Residue == r_next) & (df.Atom == 'N')]['Coors'].values[0]
            
            # Algebra
            phi = compute_dihedral(C_prev, N_curr, CA_curr, C_curr)
            psi = compute_dihedral(N_curr, CA_curr, C_curr, N_next)
            
            phi_angles.append(phi)
            psi_angles.append(psi)
        except AttributeError: # Padding por residuos faltantes
            continue
            
    return phi_angles, psi_angles

phis, psis = build_ramachandran(df_backbone)
print(f"[*] Computed {len(phis)} Dihedral Angle Pairs ($\Phi$, $\Psi$).")

### 3. Visualización Termodinámica: El Gráfico de Ramachandran
Inyectamos la salida del álgebra vectorial en un Hexbin de densidad térmica. Las colisiones termodinámicas (zonas blancas) representan colapso atómico estéricamente imposible. Las zonas cálidas (agrupamientos) confirman empíricamente la estabilidad de hélices alfa y láminas beta.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

# Hexbin for density
hb = ax.hexbin(phis, psis, gridsize=30, cmap='inferno', mincnt=1, bins='log')
ax.scatter(phis, psis, color='white', edgecolor='black', s=10, alpha=0.9)

ax.axhline(0, color='gray', linestyle='--', linewidth=1, alpha=0.5)
ax.axvline(0, color='gray', linestyle='--', linewidth=1, alpha=0.5)

ax.set_xlim(-180, 180)
ax.set_ylim(-180, 180)
ax.set_xticks(np.arange(-180, 181, 60))
ax.set_yticks(np.arange(-180, 181, 60))

ax.set_xlabel('$\\Phi$ (Phi Dihedral Angle)', fontsize=14)
ax.set_ylabel('$\\Psi$ (Psi Dihedral Angle)', fontsize=14)
ax.set_title("Gráfico Dinámico de Ramachandran (Calculado From-Scratch)", fontsize=15, pad=15)

# Annotations for Secondary Structures (Typical regions)
ax.text(-120, 120, "Láminas Beta ($\beta$)", color="cyan", fontsize=10, fontweight="bold", bbox=dict(facecolor='black', alpha=0.5))
ax.text(-60, -60, "Hélices Alfa ($\alpha_R$)", color="yellow", fontsize=10, fontweight="bold", bbox=dict(facecolor='black', alpha=0.5))
ax.text(60, 60, "$\alpha_L$", color="lime", fontsize=10, fontweight="bold", bbox=dict(facecolor='black', alpha=0.5))

plt.colorbar(hb, ax=ax, label="Log-Densidad Topológica")
plt.grid(alpha=0.1)
plt.show()